# endgame-probe -- test a candidate against eat-rest-v1 on the same seeds

Runs `external/candidates/endgame_probe.py`. It plays a candidate on fresh seeds (11000+), records a 50 s timeline of the
world and the colony plus every death and birth, and prints how the colony dies. With `--compare` it also prints the paired
per-seed score difference against another run on the same seeds, which is how a new version is judged.

Noise: the per-seed difference between two agents has a standard deviation of about 250, and the simulator is not perfectly
repeatable per seed. 16 seeds (SE ~60) only shows effects above ~+130; use 64 seeds (SE ~30) for anything to be believed.
Resumable: finished games are stored in the `--out` folder and skipped on a rerun; raising `SEEDS` only plays the new seeds.
Mechanism study: nothing is written to `results/`.

**Cluster setup:** same as `parameter-tuning.ipynb` (`.env` with `GITHUB_TOKEN=<token>`).

In [ ]:
import os

CLONE_DIR = "/home/jovyan/Nordic-AI-cup-2026"
if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"{CLONE_DIR} already cloned - skipping (use `git pull` there to update)")
else:
    # GitHub token is read from a git-ignored .env (GITHUB_TOKEN=...) in the kernel's cwd, or from the environment
    if os.path.isfile(".env"):
        for line in open(".env"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    TOKEN = os.environ.get("GITHUB_TOKEN")
    if not TOKEN:
        raise RuntimeError(f"GITHUB_TOKEN not set - create {os.path.abspath('.env')} containing GITHUB_TOKEN=<token>")
    !git clone https://{TOKEN}@github.com/sjoeen/Nordic-AI-cup-2026.git {CLONE_DIR}

In [ ]:
import glob
import os
import subprocess
import sys

os.chdir(CLONE_DIR)
!git fetch origin challenge-1V2
!git checkout challenge-1V2
!git pull origin challenge-1V2

# Must run from survival-simulator/ so `src`, `agents`, `training` import.
if os.path.basename(os.getcwd()) != "survival-simulator":
    candidates = sorted({os.path.realpath(p) for p in glob.glob(os.path.join(os.getcwd(), "**", "survival-simulator"), recursive=True)
                         if os.path.isfile(os.path.join(p, "requirements.txt"))})
    if len(candidates) != 1:
        raise RuntimeError(f"cwd is {os.getcwd()}; found {len(candidates)} survival-simulator checkouts {candidates} - %cd into the right one")
    os.chdir(candidates[0])

print("cwd:", os.getcwd())
sys.path.insert(0, os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!{sys.executable} -m pip install -r requirements.txt -r requirements-dev.txt

## 1. Baseline: eat-rest-v1 (games already stored in `logs/eg2/base` are reused)

In [ ]:
SEEDS = 64
!{sys.executable} -u external/candidates/endgame_probe.py --out logs/eg2/base --seeds {SEEDS} --brief 2>&1 | grep --line-buffered -v "pkg_resources\|pygame"

## 2. A candidate against the baseline

`CANDIDATE` is a folder under `external/candidates/` holding a `survival_agent.py` with `make_policy()` and `CONFIG`
(same shape as `original-eat-rest-preserved`). Full report first, then the paired block.

In [ ]:
CANDIDATE = "original-eat-rest-preserved"   # <- the folder to test
!{sys.executable} -u external/candidates/endgame_probe.py --out logs/eg2/{CANDIDATE} --candidate {CANDIDATE} --seeds {SEEDS} --compare logs/eg2/base 2>&1 | grep --line-buffered -v "pkg_resources\|pygame"

## Full baseline report (no new games)

In [ ]:
!{sys.executable} -u external/candidates/endgame_probe.py --out logs/eg2/base --seeds 0 2>&1 | grep --line-buffered -v "pkg_resources\|pygame"